In [1]:
import os
import pandas as pd
import numpy as np
import math
from tqdm import tqdm
import matplotlib.pyplot as plt

### Functions

In [2]:
def months_between_timestamps(dtmfunded, defaultDate):
    # get years
    int_yr_funded = dtmfunded.year
    int_yr_default = defaultDate.year
    
    # get months
    int_mo_funded = dtmfunded.month
    int_mo_default = defaultDate.month
    
    # get diff in months
    int_mo_diff = (int_yr_default - int_yr_funded) * 12 + (int_mo_default - int_mo_funded)
    
    # return
    return int_mo_diff

### Constants

In [3]:
str_dirname_output = './output'

# target
str_target = 'Early_Pay_Delinquency_60_720_Flag'

### Make output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Import the chosen targets

In [5]:
str_filename = 'df_early_indicator_targets.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
list_str_flags = list(pd.read_csv(str_local_path)['col'])
list_str_flags

['Early_Pay_Delinquency_15_60_Flag',
 'Early_Pay_Delinquency_30_90_Flag',
 'Early_Pay_Delinquency_30_180_Flag',
 'Early_Pay_Delinquency_30_360_Flag']

### Get the targets

In [6]:
list_cols = ['bigAccountId', str_target, 'defaultDate'] + list_str_flags
str_filename = 'df_targets.gzip'
str_uri = f's3://20240327-genxii-v2/02_target_creation/01_classification/{str_filename}'
df = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:283: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,bigAccountId,Early_Pay_Delinquency_60_720_Flag,defaultDate,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag
0,6,0,2008-12-03 15:01:09.283,0,0,0,0
1,25,1,1993-01-27 00:00:00.000,1,0,0,1
2,73,1,1993-01-27 00:00:00.000,0,0,0,0
3,82,1,1993-01-27 00:00:00.000,1,1,1,1
4,122,1,1993-01-27 00:00:00.000,0,0,0,1
...,...,...,...,...,...,...,...
343069,8619039,0,NaT,0,0,0,0
343070,8619083,0,NaT,0,0,0,0
343071,8619360,0,NaT,0,0,0,0
343072,8619910,0,NaT,0,0,0,0


### Import more recent PD data

In [7]:
list_cols = [
    'bigAccountId',
    'dtmfunded__app',
]
str_filename = 'df_raw.gzip'
str_uri = f's3://20240327-genxii-v2/04_pd/01_data_prep/01_data_collection/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
df_tmp

,bigAccountId,dtmfunded__app
0,5514485,2021-01-25
1,5515245,2021-01-27
2,5517730,2021-01-27
3,5514970,2021-01-28
4,5515580,2021-01-28
...,...,...
43823,6513617,2022-12-30
43824,6483306,2022-12-30
43825,6483306,2022-12-30
43826,6502220,2022-12-30


### Join

In [8]:
df = pd.merge(
    left=df_tmp,
    right=df,
    on='bigAccountId',
    how='left',
)
# show
df

,bigAccountId,dtmfunded__app,Early_Pay_Delinquency_60_720_Flag,defaultDate,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag
0,5514485,2021-01-25,0,2023-05-31 12:51:15.953,0,0,0,0
1,5515245,2021-01-27,0,NaT,0,0,0,0
2,5517730,2021-01-27,0,NaT,0,0,0,0
3,5514970,2021-01-28,0,2023-11-24 06:46:18.370,0,0,0,0
4,5515580,2021-01-28,0,NaT,0,0,0,0
...,...,...,...,...,...,...,...,...
43823,6513617,2022-12-30,0,NaT,0,0,0,0
43824,6483306,2022-12-30,0,NaT,0,0,0,0
43825,6483306,2022-12-30,0,NaT,0,0,0,0
43826,6502220,2022-12-30,1,2023-08-29 06:59:57.053,0,0,1,1


### Get months from funded to default

In [9]:
df['months_to_default'] = df.apply(
    lambda x: months_between_timestamps(
        dtmfunded=x['dtmfunded__app'],
        defaultDate=x['defaultDate'],
    ),
    axis=1,
)
df

,bigAccountId,dtmfunded__app,Early_Pay_Delinquency_60_720_Flag,defaultDate,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,months_to_default
0,5514485,2021-01-25,0,2023-05-31 12:51:15.953,0,0,0,0,28.0
1,5515245,2021-01-27,0,NaT,0,0,0,0,NaN
2,5517730,2021-01-27,0,NaT,0,0,0,0,NaN
3,5514970,2021-01-28,0,2023-11-24 06:46:18.370,0,0,0,0,34.0
4,5515580,2021-01-28,0,NaT,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...
43823,6513617,2022-12-30,0,NaT,0,0,0,0,NaN
43824,6483306,2022-12-30,0,NaT,0,0,0,0,NaN
43825,6483306,2022-12-30,0,NaT,0,0,0,0,NaN
43826,6502220,2022-12-30,1,2023-08-29 06:59:57.053,0,0,1,1,8.0


### Make a tag for default at 24 (1) or not (0)

In [10]:
df['bitDefault'] = df['months_to_default'].apply(
    lambda x: 1 if x <= 24 else 0,
)
# drop
list_cols = [
    'dtmfunded__app',
    'defaultDate',
    'months_to_default',
]
df.drop(list_cols, axis=1, inplace=True)
# show
df

,bigAccountId,Early_Pay_Delinquency_60_720_Flag,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,bitDefault
0,5514485,0,0,0,0,0,0
1,5515245,0,0,0,0,0,0
2,5517730,0,0,0,0,0,0
3,5514970,0,0,0,0,0,0
4,5515580,0,0,0,0,0,0
...,...,...,...,...,...,...,...
43823,6513617,0,0,0,0,0,0
43824,6483306,0,0,0,0,0,0
43825,6483306,0,0,0,0,0,0
43826,6502220,1,0,0,1,1,1


### Get the conditional probabilities

In [11]:
# 1
list_cols = [col for col in df.columns if col not in ['bigAccountId', str_target, 'bitDefault']]
list_cols = list_cols + [str_target, 'bitDefault']

list_dict_row = []
for col in tqdm(list_cols):
    df_tmp = df[df[col] == 1].copy()
    # get means of the other columns
    ser_mean = df_tmp[list_cols].mean()
    dict_mean = dict(ser_mean)
    dict_row = {'col': col}
    dict_row = {**dict_row, **dict_mean}
    # append
    list_dict_row.append(dict_row)
df_tmp = pd.DataFrame(list_dict_row)
df_tmp

100%|██████████| 6/6 [00:00<00:00, 445.66it/s]


,col,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,bitDefault
0,Early_Pay_Delinquency_15_60_Flag,1.000000,0.584812,0.784972,0.897202,0.802718,0.530935
1,Early_Pay_Delinquency_30_90_Flag,0.709878,1.000000,1.000000,1.000000,0.893266,0.623132
2,Early_Pay_Delinquency_30_180_Flag,0.411430,0.431792,1.000000,1.000000,0.853109,0.552371
3,Early_Pay_Delinquency_30_360_Flag,0.273756,0.251366,0.582146,1.000000,0.799902,0.479171
4,Early_Pay_Delinquency_60_720_Flag,0.244022,0.223707,0.494800,0.796948,1.000000,0.563618
5,bitDefault,0.285997,0.276524,0.567689,0.845935,0.998708,1.000000


### Save

In [12]:
str_filename = 'df_conditionals.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)